<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Homework/capstone/capstone-ames_housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone: What Makes a House Sell for More?

**DSC 210 Foundations of Data Science**

Data: De Cock, D. (2011), *Ames, Iowa: Alternative to the Boston Housing Data as an End of Semester Regression Project*, Journal of Statistics Education 19(3). 2,930 residential sales, Ames, Iowa, 2006 to 2010.

```
ASK  ->  GET  ->  EXPLORE  ->  MODEL  ->  COMMUNICATE
```

*Last major revision: 2026-08-19*

---

## The brief

A regional assessor's office has five years of residential sales records and a question:

> **"What actually drives what a house sells for, and how accurately can we predict a sale price before it happens?"**

You will take that from a vague sentence to a defended answer, using the full arc of the course. Nothing here is a new technique. Everything here is something you have already done once, now on a dataset big enough and messy enough to be worth doing properly.

## How to work

The parts run in order and later parts depend on earlier ones. Each part ends with written questions; **answer them in sentences**, because the memo in Part 9 is assembled from them.

Give yourself time on Parts 4 and 5. Exploration is where the findings come from; modelling only confirms them.

---
## Part 1. Load and Look
---

In [ ]:
# TASK 1a. Load, and run the standard first look.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

ames_full = pd.read_csv('https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/csv_data/AmesHousing.csv')

print('shape:', ____)
print('columns:', ames_full.shape[1])

Eighty-two columns is more than we need. Real projects begin by choosing a working subset, and that choice is itself an analytical decision.

We will use eighteen columns chosen to span every kind of feature you have studied.

In [ ]:
# TASK 1b. Select the working columns and tidy the names.
KEEP = ['SalePrice', 'Gr Liv Area', 'Lot Area', 'Lot Frontage', 'Total Bsmt SF',
        'Garage Area', 'Year Built', 'Year Remod/Add', 'Overall Qual',
        'Kitchen Qual', 'Exter Qual', 'Neighborhood', 'House Style',
        'Central Air', 'Garage Type', 'Fireplaces', 'Full Bath', 'Bedroom AbvGr']

ames = ames_full[KEEP].copy()
ames.columns = [c.lower().replace(' ', '_').replace('/', '_') for c in ames.columns]

print(list(ames.columns))
ames.____()

**The data dictionary** for our eighteen columns:

| Column | Meaning |
| --- | --- |
| `saleprice` | sale price in dollars |
| `gr_liv_area` | above-ground living area, square feet |
| `lot_area` | lot size, square feet |
| `lot_frontage` | feet of street frontage |
| `total_bsmt_sf` | basement area, square feet |
| `garage_area` | garage area, square feet |
| `year_built` | year of construction |
| `year_remod_add` | year of last remodel (equals `year_built` if none) |
| `overall_qual` | overall material and finish, 1 (very poor) to 10 (excellent) |
| `kitchen_qual`, `exter_qual` | quality: Ex, Gd, TA (typical), Fa, Po |
| `neighborhood` | one of 28 Ames neighborhoods |
| `house_style` | 1Story, 2Story, SLvl, and so on |
| `central_air` | Y or N |
| `garage_type` | Attchd, Detchd, BuiltIn, and so on |
| `fireplaces`, `full_bath`, `bedroom_abvgr` | counts |

---
## Part 2. Feature Types
---

Before computing anything, decide what each column *is*. This determines every tool you are permitted to use.

In [ ]:
# TASK 2a. Evidence for the audit.
for col in ['overall_qual', 'kitchen_qual', 'exter_qual', 'central_air',
            'house_style', 'garage_type', 'neighborhood']:
    print(f'{col:16} {ames[col].nunique():>3} distinct')
print()
print('kitchen_qual:', ames['kitchen_qual'].value_counts().to_dict())
print('overall_qual range:', ames['overall_qual'].min(), 'to', ames['overall_qual'].max())

#### **Task 2b.** Complete the audit.

| Column | Stored as | Measurement scale | A meaningful summary |
| --- | --- | --- | --- |
| `saleprice` |  |  |  |
| `gr_liv_area` |  |  |  |
| `year_built` |  |  |  |
| `overall_qual` |  |  |  |
| `kitchen_qual` |  |  |  |
| `neighborhood` |  |  |  |
| `central_air` |  |  |  |
| `fireplaces` |  |  |  |

#### **Task 2c.** Answer in sentences.

**A.** `overall_qual` is stored as an integer from 1 to 10 and `kitchen_qual` as text from Po to Ex. Are they the same measurement scale? What does that say about relying on the stored type?

**B.** Which of these columns can you take a mean of, and which only a median or a mode? Give one example of each.

**C.** `year_built` is a number you can subtract but not sensibly halve. Name its scale and say what property it lacks.

**D.** To use `neighborhood` in a model you would have to turn 28 text labels into numbers. Explain why simply numbering them 1 to 28 would be a mistake.

---
## Part 3. Missing Data
---

In [ ]:
# TASK 3a. Where is data missing?
missing = ames.isna().sum()
print(missing[missing > 0].sort_values(ascending=False).to_string())

Two columns have substantial gaps, and they are **not the same kind of gap**. Investigate before deciding anything.

In [ ]:
# TASK 3b. Are the garage_type gaps really missing?
no_type = ames['garage_type'].isna()

print('houses with no garage_type:', no_type.sum())
print('of those, garage_area is 0 or missing:',
      (ames.loc[no_type, 'garage_area'].fillna(0) == ____).sum())

#### **Task 3c.** Answer, then act.

**A.** Every house with a missing `garage_type` has a garage area of zero. What does the missing value actually mean? Is this data that was lost, or data that was never applicable?

**B.** `lot_frontage` is missing for 490 houses, but `lot_area` is present for all of them. Is this the same kind of gap as `garage_type`? Explain the difference.

**C.** For each of the two columns, choose a treatment and defend it: drop the rows, fill with a value, or encode the absence as its own category.

**D.** Module 8's rule was that there is no such thing as clean, only clean enough for a purpose. State the purpose you are cleaning for.

In [ ]:
# TASK 3d. Act on your decisions.
# garage_type: the absence is information. Encode it rather than discarding it.
ames['has_garage'] = (~ames['garage_type'].isna()).astype(____)

# The two stray gaps in area columns are true blanks; fill with 0 (no basement / no garage).
ames['total_bsmt_sf'] = ames['total_bsmt_sf'].fillna(0)
ames['garage_area']   = ames['garage_area'].fillna(0)

print('rows:', ames.shape[0])
print('remaining missing:', ames.isna().sum()[ames.isna().sum() > 0].to_dict())

---
## Part 4. Exploring One Variable at a Time
---

In [ ]:
# TASK 4a. The target.
sns.histplot(data=ames, x=____, bins=40)
plt.title('Sale price')
plt.show()

print('mean  : $', round(ames['saleprice'].mean()))
print('median: $', round(ames['saleprice'].____()))
print('skew  :', round(ames['saleprice'].skew(), 2))

In [ ]:
# TASK 4b. Module 5 offered a repair for exactly this shape.
ames['log_price'] = np.log(ames['saleprice'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(data=ames, x='saleprice', bins=40, ax=axes[0])
axes[0].set_title(f"raw price (skew {ames['saleprice'].skew():.2f})")
sns.histplot(data=ames, x=____, bins=40, ax=axes[1])
axes[1].set_title(f"log price (skew {ames['log_price'].skew():.2f})")
plt.show()

In [ ]:
# TASK 4c. A second numeric variable, and an ordinal one.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=ames, x='gr_liv_area', bins=40, ax=axes[0])
axes[0].set_title('above-ground living area')
sns.countplot(data=ames, x='overall_qual', ax=axes[1])
axes[1].set_title('overall quality rating')
plt.show()

print(ames['gr_liv_area'].describe().round(0).to_string())

#### **Task 4d.** Answer in sentences.

**A.** Is sale price skewed, and in which direction? Which of mean or median describes a typical Ames house better, and by how many dollars do they differ?

**B.** The log transform changed the skew from about 1.74 to about 0.00. Explain what taking a logarithm did to the long right tail.

**C.** Living area also has a long right tail. Say what those extreme values are likely to be, and whether you would remove them.

**D.** `overall_qual` is heavily concentrated at 5 and 6. What does that concentration mean for how much a model can learn about very poor and very excellent houses?

---
## Part 5. Exploring Relationships
---

In [ ]:
# TASK 5a. The headline relationship.
sns.scatterplot(data=ames, x='gr_liv_area', y=____, alpha=0.4)
plt.xlabel('above-ground living area (sq ft)'); plt.ylabel('sale price ($)')
plt.title('Price against size')
plt.show()

print('correlation:', round(ames['gr_liv_area'].corr(ames['saleprice']), 3))

In [ ]:
# TASK 5b. Does quality change that relationship? Add a third variable with hue.
sns.scatterplot(data=ames, x='gr_liv_area', y='saleprice',
                hue=____, palette='viridis', alpha=0.6)
plt.xlabel('living area (sq ft)'); plt.ylabel('sale price ($)')
plt.title('Price against size, coloured by overall quality')
plt.show()

In [ ]:
# TASK 5c. Does an ordinal feature order the prices the way it claims to?
order = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
sns.boxplot(data=ames, x='kitchen_qual', y='saleprice', order=____)
plt.xlabel('kitchen quality'); plt.ylabel('sale price ($)')
plt.title('Price by kitchen quality')
plt.show()

print(ames.groupby('kitchen_qual')['saleprice'].agg(['count', 'median']).round(0).to_string())

In [ ]:
# TASK 5d. Which numeric features move with price?
NUMERIC = ['saleprice', 'gr_liv_area', 'lot_area', 'total_bsmt_sf', 'garage_area',
           'year_built', 'overall_qual', 'fireplaces', 'full_bath', 'bedroom_abvgr']

corr = ames[NUMERIC].corr().round(2)
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap='vlag', vmin=-1, vmax=1, annot_kws={'size': 7})
plt.title('Correlations among numeric features')
plt.show()

print(corr['saleprice'].sort_values(ascending=False).to_string())

#### **Task 5e.** Write down your findings. These become the memo.

**A.** Report the correlation between living area and price. Describe the scatterplot in words: is the relationship linear, and does the spread stay constant as size increases?

**B.** In Task 5b the colours are not scattered randomly. Describe the pattern, and explain what it means for a model given size *and* quality rather than size alone.

**C.** From Task 5c, does kitchen quality order the prices as its ordinal scale claims? Look at the count for `Po`. What does Module 7 say about group summaries resting on very few observations?

**D.** From the correlation matrix, name the three features most strongly related to price. Then name one pair of *features* strongly correlated with each other, and say why that matters when interpreting a model.

**E.** `bedroom_abvgr` has a surprisingly weak correlation with price. Give a plausible explanation that refers to another column.

---
## Part 6. Structure Without the Label
---

In [ ]:
# TASK 6a. Build a numeric matrix. Encode the ordinal columns in their real order.
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

qmap = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
ames['kitchen_ord'] = ames['kitchen_qual'].map(qmap)
ames['exter_ord']   = ames['exter_qual'].map(qmap)
ames['central_air_y'] = (ames['central_air'] == 'Y').astype(int)

FEATURES = ['gr_liv_area', 'lot_area', 'total_bsmt_sf', 'garage_area', 'year_built',
            'year_remod_add', 'overall_qual', 'kitchen_ord', 'exter_ord',
            'fireplaces', 'full_bath', 'bedroom_abvgr', 'has_garage', 'central_air_y']

model_df = ames.dropna(subset=FEATURES + ['saleprice']).copy()
print('rows available for modelling:', model_df.shape[0])

# PCA and k-means both require one preparation step. Which one?
Z = ____().fit_transform(model_df[FEATURES])

In [ ]:
# TASK 6b. PCA: can fourteen columns be summarized by two?
pca = PCA(n_components=2).fit(Z)
components = pca.transform(Z)

print('variance explained by PC1:', round(pca.explained_variance_ratio_[0], 3))
print('variance explained by PC2:', round(pca.explained_variance_ratio_[1], 3))
print('together                 :', round(pca.explained_variance_ratio_.sum(), 3))
print()
loadings = pd.DataFrame(pca.components_.T, index=FEATURES, columns=['PC1', 'PC2']).round(2)
print(loadings.to_string())

In [ ]:
# TASK 6c. Cluster the houses, then reveal the price the algorithm never saw.
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z)
model_df['cluster'] = km.labels_

print(model_df.groupby('cluster')[['gr_liv_area', 'overall_qual', 'year_built', 'saleprice']]
      .median().round(0).to_string())

sns.scatterplot(x=components[:, 0], y=components[:, 1], hue=____,
                palette='viridis', s=12, alpha=0.7)
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('Ames houses in two principal components')
plt.show()

#### **Task 6d.** Answer in sentences.

**A.** Why did the data have to be standardized before PCA and k-means? Name a specific pair of columns in `FEATURES` that would otherwise dominate.

**B.** Read the PC1 loadings. Are they mostly one sign or mixed? Propose a name for PC1 and defend it from the loadings.

**C.** Give each of the three clusters a short descriptive name based on its median size, quality, and year.

**D.** Median price differs sharply across clusters, yet price was never given to the algorithm. Does that prove the clustering found something real? Be careful, and connect your answer to what the clustering *was* given.

**E.** PC1 and PC2 together explain only about half the variance. Is that a failure? What would you have to give up to capture 90%?

---
## Part 7. Predicting Price
---

In [ ]:
# TASK 7a. Split, and establish the baseline before any model.
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = model_df[FEATURES]
y = model_df['saleprice']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=210)

baseline = np.full(len(y_test), y_train.mean())
print('baseline MAE: $', round(mean_absolute_error(y_test, ____)))

In [ ]:
# TASK 7b. Four models. Exactly one of them needs scaling; make sure it gets it.
scaler = StandardScaler().fit(X_train)
Xtr_s, Xte_s = scaler.transform(X_train), scaler.transform(X_test)

results = []
for name, model, needs_scaling in [
        ('linear regression', LinearRegression(), False),
        ('k-NN, k = 10',      KNeighborsRegressor(n_neighbors=10), ____),
        ('tree, depth 6',     DecisionTreeRegressor(max_depth=6, random_state=0), False),
        ('random forest',     RandomForestRegressor(n_estimators=300, random_state=0), False)]:
    A, B = (Xtr_s, Xte_s) if needs_scaling else (X_train, X_test)
    model.fit(A, y_train)
    p = model.predict(B)
    results.append((name, mean_absolute_error(y_test, p),
                    mean_squared_error(y_test, p) ** 0.5, r2_score(y_test, p)))

print(f"{'model':20}{'MAE':>10}{'RMSE':>10}{'R2':>8}")
for n, mae, rmse, r2 in results:
    print(f'{n:20}{mae:>10.0f}{rmse:>10.0f}{r2:>8.3f}')

In [ ]:
# TASK 7c. Cross-validate, so the comparison does not rest on one lucky split.
kf = KFold(5, shuffle=True, random_state=210)

for name, model in [('linear regression', LinearRegression()),
                    ('random forest', RandomForestRegressor(n_estimators=300, random_state=0))]:
    s = cross_val_score(model, X, y, cv=____, scoring='r2')
    print(f'{name:20} mean R2 = {s.mean():.3f}   folds {np.round(s, 3)}')

In [ ]:
# TASK 7d. Diagnose the best model: residuals and importances.
best = RandomForestRegressor(n_estimators=300, random_state=0).fit(X_train, y_train)
pred = best.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.scatterplot(x=pred, y=y_test - pred, alpha=0.5, ax=axes[0])
axes[0].axhline(0, color='crimson', linestyle='--')
axes[0].set_xlabel('predicted price ($)'); axes[0].set_ylabel('residual ($)')
axes[0].set_title('Residuals')

imp = pd.Series(best.feature_importances_, index=FEATURES).sort_values()
sns.barplot(x=imp.values, y=imp.index, orient='h', ax=axes[1])
axes[1].set_title('Feature importance'); axes[1].set_xlabel('')
plt.tight_layout(); plt.show()

print(imp.sort_values(ascending=False).round(3).head(6).to_string())

#### **Task 7e.** Answer in sentences.

**A.** Rank the four models by MAE. State the best model's error in dollars, and say how much better than the baseline that is.

**B.** For every model, RMSE exceeds MAE. Explain what that gap tells you about the distribution of the errors.

**C.** The forest beats the line. Using a specific finding from Part 5, explain why that was predictable.

**D.** Does cross-validation change the ranking you got from the single split? Report both means and say which number you would put in a report, and why.

**E.** One feature dominates the importance chart. Name it, and connect it to Part 5's correlation matrix. Then give one reason to be careful reading importances, referring to your answer to Task 5d.

**F.** Read the residual plot. Where are the errors largest, and what does that say about which houses this model serves worst?

---
## Part 8. The Same Question as a Classification
---

In [ ]:
# TASK 8a. Turn price into a category and check the balance.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

model_df['expensive'] = (model_df['saleprice'] > model_df['saleprice'].median()).astype(int)
print(model_df['expensive'].value_counts().to_dict())

Xc, yc = model_df[FEATURES], model_df['expensive']
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25,
                                              random_state=210, stratify=____)

In [ ]:
# TASK 8b. Two classifiers, against the baseline.
sc = StandardScaler().fit(Xc_tr)

logit = LogisticRegression(max_iter=5000).fit(sc.transform(Xc_tr), yc_tr)
forest_c = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xc_tr, yc_tr)

print('baseline      :', round(max(yc_te.mean(), 1 - yc_te.mean()), 3))
print('logistic      :', round(logit.score(sc.transform(Xc_te), yc_te), 3))
print('random forest :', round(forest_c.score(Xc_te, yc_te), 3))
print()
print(classification_report(yc_te, forest_c.predict(Xc_te),
                            target_names=['affordable', 'expensive']))

In [ ]:
# TASK 8c. The confusion matrix.
cm = confusion_matrix(yc_te, forest_c.predict(Xc_te))
sns.heatmap(cm, annot=True, fmt='d', cbar=False, cmap='Blues',
            xticklabels=['affordable', 'expensive'],
            yticklabels=['affordable', 'expensive'])
plt.xlabel('predicted'); plt.ylabel('actual')
plt.title('Above or below median price')
plt.show()

#### **Task 8d.** Answer in sentences.

**A.** Why is the baseline almost exactly 0.50 here? What did splitting at the median guarantee?

**B.** Report precision and recall for the `expensive` class. Are they close? What would it mean if one were much lower than the other?

**C.** The classifier reaches roughly 92% accuracy while the regression explains about 88% of the variance. Are those two numbers measuring the same thing? Explain why they cannot be compared directly.

**D.** You converted a dollar amount into two categories. Name one thing that was lost. Then describe a situation in which the assessor's office would genuinely prefer the category to the number.

**E.** Which would you deliver: the regression from Part 7 or the classifier from Part 8? Defend the choice by reference to the brief at the top of this notebook.

---
## Part 9. The Memo
---

Everything above was analysis. This is the deliverable.

#### **Task 9a. The memo**

In a markdown cell, write **500 to 700 words** to the director of the assessor's office. She is intelligent, busy, and not a data scientist. She will not read code.

Include, clearly labelled:

**1. The question, restated.** How you turned "what drives price" into something answerable.

**2. Four findings.** One sentence each, each backed by a specific number from your own work, each stated in real units (dollars, square feet, quality ratings). At least one must come from Part 5, and at least one must be something a real estate professional would find non-obvious.

**3. The model.** Which one you would put into service, its typical error **in dollars**, and how that compares with predicting the average every time.

**4. Two limitations.** One about the data and one about the model. At least one must be something you discovered rather than something you were told.

**5. One recommendation and one next question.**

**Constraints.** No code. No unexplained jargon: if you write $R^2$, define it in the same sentence. No causal claims from associational evidence.

#### **Task 9b. Reflection**

**A.** Name the part of this capstone where you learned the most about the *data* rather than about a technique.

**B.** Pick one tool from the semester's toolkit that you did **not** use here. What would have had to be different about the data or the question for it to apply?

**C.** If you had two more weeks and could collect one additional column, what would it be, and which finding would it sharpen?